# Named Entity Recognition System

Build a sequence labeling system for CoNLL-2003 named entity recognition. The workflow follows the same structure as the Consumer Complaint project: load data, inspect it, preprocess it, train multiple deep learning models, fine-tune a transformer, evaluate, compare, save checkpoints, and test on new examples.

## 1. Imports and Configuration

In [1]:
from pathlib import Path
from collections import Counter
import gzip
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset, load_from_disk
from seqeval.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import DataCollatorForTokenClassification, Trainer, TrainerCallback, TrainingArguments
from transformers import get_linear_schedule_with_warmup

try:
    from torchcrf import CRF
except ImportError:
    CRF = None

PROJECT_DIR = Path.cwd()
DATASET_DIR = PROJECT_DIR / 'conll2003_dataset'
EMBEDDINGS_DIR = PROJECT_DIR / 'embeddings'
CHECKPOINT_DIR = PROJECT_DIR / 'checkpoints'
EMBEDDINGS_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CUSTOM_HYPERPARAMS = {
    'embedding_dim': 256,
    'embedding_source': 'GloVe',
    'use_pretrained_embeddings': True,
    'pretrained_embedding_path': str(EMBEDDINGS_DIR / 'glove.6B.300d.txt'),
    'freeze_embeddings': False,
    'char_embedding_dim': 32,
    'char_hidden_size': 64,
    'hidden_size': 512,
    'num_layers': 2,
    'batch_size': 128,
    'max_sequence_length': 128,
    'max_char_length': 24,
    'learning_rate': 5e-4,
    'max_epochs': 50,
    'dropout': 0.3,
    'weight_decay': 1e-4,
    'gradient_clip': 1.0,
    'early_stopping_patience': 3,
    'min_improvement': 0.001,
    'bidirectional': True,
}

TRANSFORMER_HYPERPARAMS = {
    'base_model': 'bert-base-cased',
    'max_length': 256,
    'batch_size': 32,
    'learning_rate': 2e-5,
    'weight_decay': 0.01,
    'max_epochs': 5,
    'early_stopping_patience': 2,
    'warmup_ratio': 0.1,
    'mixed_precision_amp': True,
    'optimizer': 'AdamW',
}

DEVICE

c:\Users\MOS3AD\miniconda3\envs\subway_rl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda')

## 2. Load Dataset

In [2]:
if DATASET_DIR.exists():
    dataset = load_from_disk(str(DATASET_DIR))
else:
    dataset = load_dataset('eriktks/conll2003', cache_dir=str(PROJECT_DIR / 'hf_cache'))
    dataset.save_to_disk(str(DATASET_DIR))

dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

## 3. Dataset Checks

In [3]:
split_summary = pd.DataFrame(
    [{'split': split, 'rows': len(dataset[split]), 'columns': len(dataset[split].column_names)} for split in dataset]
)
split_summary

,split,rows,columns
0,train,14041,5
1,validation,3250,5
2,test,3453,5


In [4]:
label_names = dataset['train'].features['ner_tags'].feature.names
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in id2label.items()}
id2label

{0: 'O',
 1: 'B-PER',
 2: 'I-PER',
 3: 'B-ORG',
 4: 'I-ORG',
 5: 'B-LOC',
 6: 'I-LOC',
 7: 'B-MISC',
 8: 'I-MISC'}

In [5]:
def decode_tags(example):
    return {'ner_labels': [id2label[tag_id] for tag_id in example['ner_tags']]}

dataset = dataset.map(decode_tags)
pd.DataFrame(dataset['train'][:5])

,id,tokens,pos_tags,chunk_tags,ner_tags,ner_labels
0,0,"[EU, rejects, German, call, to, boycott, Briti...","[22, 42, 16, 21, 35, 37, 16, 21, 7]","[11, 21, 11, 12, 21, 22, 11, 12, 0]","[3, 0, 7, 0, 0, 0, 7, 0, 0]","[B-ORG, O, B-MISC, O, O, O, B-MISC, O, O]"
1,1,"[Peter, Blackburn]","[22, 22]","[11, 12]","[1, 2]","[B-PER, I-PER]"
2,2,"[BRUSSELS, 1996-08-22]","[22, 11]","[11, 12]","[5, 0]","[B-LOC, O]"
3,3,"[The, European, Commission, said, on, Thursday...","[12, 22, 22, 38, 15, 22, 28, 38, 15, 16, 21, 3...","[11, 12, 12, 21, 13, 11, 11, 21, 13, 11, 12, 1...","[0, 3, 4, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, ...","[O, B-ORG, I-ORG, O, O, O, O, O, O, B-MISC, O,..."
4,4,"[Germany, 's, representative, to, the, Europea...","[22, 27, 21, 35, 12, 22, 22, 27, 16, 21, 22, 2...","[11, 11, 12, 13, 11, 12, 12, 11, 12, 12, 12, 1...","[5, 0, 0, 0, 0, 3, 4, 0, 0, 0, 1, 2, 0, 0, 0, ...","[B-LOC, O, O, O, O, B-ORG, I-ORG, O, O, O, B-P..."


## 4. Entity Distribution

In [6]:
tag_counter = Counter()
for tags in dataset['train']['ner_tags']:
    tag_counter.update(tags)

tag_distribution = pd.DataFrame(
    [{'label': id2label[tag_id], 'count': count} for tag_id, count in tag_counter.items()]
).sort_values('count', ascending=False)
tag_distribution

,label,count
1,O,169578
5,B-LOC,7140
3,B-PER,6600
0,B-ORG,6321
4,I-PER,4528
6,I-ORG,3704
2,B-MISC,3438
8,I-LOC,1157
7,I-MISC,1155


## 5. Transformer Tokenization and Label Alignment

In [7]:
TRANSFORMER_MODEL_NAME = TRANSFORMER_HYPERPARAMS['base_model']
MAX_LENGTH = TRANSFORMER_HYPERPARAMS['max_length']
tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LENGTH,
    )

    aligned_labels = []
    for batch_index, labels in enumerate(examples['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=batch_index)
        previous_word_id = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(labels[word_id])
            else:
                label_ids.append(labels[word_id])
            previous_word_id = word_id
        aligned_labels.append(label_ids)

    tokenized['labels'] = aligned_labels
    return tokenized

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'ner_labels', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'ner_labels', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'ner_labels', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})

## 6. Custom Model Preparation

Build word and character vocabularies here for LSTM, BiLSTM, and BiLSTM + CRF models. Use `<PAD>` for padding and `<UNK>` for OOV tokens.

In [8]:
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'

word_counter = Counter(token.lower() for row in dataset['train']['tokens'] for token in row)
vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for word, _ in word_counter.most_common(50000 - len(vocab)):
    vocab[word] = len(vocab)

len(vocab)


21011

## 7. Load GloVe Word Embeddings

This code loads a pre-trained GloVe `.txt` file and creates the embedding matrix used by the LSTM, BiLSTM, and BiLSTM + CRF models.

In [9]:
def open_embedding_file(path):
    path = Path(path)
    if path.suffix == '.gz':
        return gzip.open(path, 'rt', encoding='utf-8', errors='ignore')
    return open(path, 'r', encoding='utf-8', errors='ignore')

def build_pretrained_embedding_matrix(vocab, embedding_path):
    embedding_path = Path(embedding_path)
    if not embedding_path.exists():
        raise FileNotFoundError(
            f'GloVe file not found: {embedding_path}. '
            'Place glove.6B.300d.txt inside the embeddings/ folder.'
        )

    vectors = {}
    embedding_dim = None
    with open_embedding_file(embedding_path) as file:
        for line_number, line in enumerate(file):
            parts = line.rstrip().split()
            if len(parts) <= 2:
                continue
            if line_number == 0 and len(parts) == 2 and parts[0].isdigit() and parts[1].isdigit():
                continue
            word = parts[0]
            values = parts[1:]
            if embedding_dim is None:
                embedding_dim = len(values)
            if len(values) != embedding_dim or word.lower() not in vocab:
                continue
            vectors[word.lower()] = np.asarray(values, dtype=np.float32)

    if embedding_dim is None:
        raise ValueError(f'Could not read vectors from {embedding_path}')

    matrix = np.random.normal(0, 0.05, size=(len(vocab), embedding_dim)).astype(np.float32)
    matrix[vocab[PAD_TOKEN]] = np.zeros(embedding_dim, dtype=np.float32)
    found = 0
    for word, index in vocab.items():
        vector = vectors.get(word)
        if vector is not None:
            matrix[index] = vector
            found += 1

    coverage = found / max(len(vocab) - 2, 1)
    CUSTOM_HYPERPARAMS['embedding_dim'] = embedding_dim
    CUSTOM_HYPERPARAMS['pretrained_embedding_path'] = str(embedding_path)
    CUSTOM_HYPERPARAMS['pretrained_embedding_coverage'] = coverage
    print(f'Loaded {found:,}/{len(vocab):,} pretrained vectors from {embedding_path.name} ({coverage:.2%} coverage).')
    return torch.tensor(matrix, dtype=torch.float32)

pretrained_embedding_matrix = None
if CUSTOM_HYPERPARAMS['use_pretrained_embeddings']:
    pretrained_embedding_matrix = build_pretrained_embedding_matrix(
        vocab,
        CUSTOM_HYPERPARAMS['pretrained_embedding_path'],
    )

embedding_status = {
    'path': CUSTOM_HYPERPARAMS['pretrained_embedding_path'],
    'loaded': pretrained_embedding_matrix is not None,
    'shape': tuple(pretrained_embedding_matrix.shape) if pretrained_embedding_matrix is not None else None,
    'freeze_embeddings': CUSTOM_HYPERPARAMS['freeze_embeddings'],
}
embedding_status

Loaded 18,415/21,011 pretrained vectors from glove.6B.300d.txt (87.65% coverage).


{'path': 'd:\\NLP_INSTANT\\Named Entity Recognition Project\\embeddings\\glove.6B.300d.txt',
 'loaded': True,
 'shape': (21011, 300),
 'freeze_embeddings': False}

## 8. LSTM, BiLSTM, and BiLSTM + CRF

Implement the recurrent architectures in this section. Keep training loops consistent so model comparisons are fair.

In [10]:
char_counter = Counter(char for row in dataset['train']['tokens'] for token in row for char in token)
char_vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for char, _ in char_counter.most_common():
    char_vocab[char] = len(char_vocab)

class CharBiLSTMEncoder(nn.Module):
    def __init__(self, char_vocab_size, char_embedding_dim=32, char_hidden_size=64, char_pad_id=0):
        super().__init__()
        self.char_embedding = nn.Embedding(char_vocab_size, char_embedding_dim, padding_idx=char_pad_id)
        self.char_lstm = nn.LSTM(char_embedding_dim, char_hidden_size, batch_first=True, bidirectional=True)
        self.output_dim = char_hidden_size * 2

    def forward(self, char_ids):
        batch_size, seq_len, char_len = char_ids.shape
        flat_char_ids = char_ids.view(batch_size * seq_len, char_len)
        embedded = self.char_embedding(flat_char_ids)
        _, (hidden, _) = self.char_lstm(embedded)
        char_features = torch.cat([hidden[-2], hidden[-1]], dim=-1)
        return char_features.view(batch_size, seq_len, self.output_dim)

class LSTMEncoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        char_vocab_size,
        word_embedding_dim=256,
        char_embedding_dim=32,
        char_hidden_size=64,
        hidden_size=512,
        num_layers=2,
        bidirectional=False,
        dropout=0.3,
        pretrained_embedding_matrix=None,
        freeze_embeddings=False,
    ):
        super().__init__()
        if pretrained_embedding_matrix is not None:
            self.word_embedding = nn.Embedding.from_pretrained(
                pretrained_embedding_matrix,
                freeze=freeze_embeddings,
                padding_idx=0,
            )
            word_embedding_dim = pretrained_embedding_matrix.shape[1]
        else:
            self.word_embedding = nn.Embedding(vocab_size, word_embedding_dim, padding_idx=0)
        self.char_encoder = CharBiLSTMEncoder(char_vocab_size, char_embedding_dim, char_hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            word_embedding_dim + self.char_encoder.output_dim,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )

    def forward(self, word_ids, char_ids):
        word_features = self.word_embedding(word_ids)
        char_features = self.char_encoder(char_ids)
        features = self.dropout(torch.cat([word_features, char_features], dim=-1))
        encoded, _ = self.lstm(features)
        return self.dropout(encoded)

class LSTMTagger(nn.Module):
    def __init__(self, vocab_size, char_vocab_size, num_labels, hyperparams, pretrained_embedding_matrix=None):
        super().__init__()
        self.encoder = LSTMEncoder(
            vocab_size,
            char_vocab_size,
            word_embedding_dim=hyperparams['embedding_dim'],
            char_embedding_dim=hyperparams['char_embedding_dim'],
            char_hidden_size=hyperparams['char_hidden_size'],
            hidden_size=hyperparams['hidden_size'],
            num_layers=hyperparams['num_layers'],
            bidirectional=False,
            dropout=hyperparams['dropout'],
            pretrained_embedding_matrix=pretrained_embedding_matrix,
            freeze_embeddings=hyperparams['freeze_embeddings'],
        )
        self.classifier = nn.Linear(hyperparams['hidden_size'], num_labels)

    def forward(self, word_ids, char_ids):
        return self.classifier(self.encoder(word_ids, char_ids))

class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, char_vocab_size, num_labels, hyperparams, pretrained_embedding_matrix=None):
        super().__init__()
        self.encoder = LSTMEncoder(
            vocab_size,
            char_vocab_size,
            word_embedding_dim=hyperparams['embedding_dim'],
            char_embedding_dim=hyperparams['char_embedding_dim'],
            char_hidden_size=hyperparams['char_hidden_size'],
            hidden_size=hyperparams['hidden_size'],
            num_layers=hyperparams['num_layers'],
            bidirectional=True,
            dropout=hyperparams['dropout'],
            pretrained_embedding_matrix=pretrained_embedding_matrix,
            freeze_embeddings=hyperparams['freeze_embeddings'],
        )
        self.classifier = nn.Linear(hyperparams['hidden_size'] * 2, num_labels)

    def forward(self, word_ids, char_ids):
        return self.classifier(self.encoder(word_ids, char_ids))

class BiLSTMCRFTagger(nn.Module):
    def __init__(self, vocab_size, char_vocab_size, num_labels, hyperparams, pretrained_embedding_matrix=None):
        super().__init__()
        if CRF is None:
            raise ImportError('Install pytorch-crf to use BiLSTMCRFTagger: pip install pytorch-crf')
        self.encoder = LSTMEncoder(
            vocab_size,
            char_vocab_size,
            word_embedding_dim=hyperparams['embedding_dim'],
            char_embedding_dim=hyperparams['char_embedding_dim'],
            char_hidden_size=hyperparams['char_hidden_size'],
            hidden_size=hyperparams['hidden_size'],
            num_layers=hyperparams['num_layers'],
            bidirectional=True,
            dropout=hyperparams['dropout'],
            pretrained_embedding_matrix=pretrained_embedding_matrix,
            freeze_embeddings=hyperparams['freeze_embeddings'],
        )
        self.emissions = nn.Linear(hyperparams['hidden_size'] * 2, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, word_ids, char_ids, mask):
        emissions = self.emissions(self.encoder(word_ids, char_ids))
        return self.crf.decode(emissions, mask=mask.bool())

    def loss(self, word_ids, char_ids, labels, mask):
        emissions = self.emissions(self.encoder(word_ids, char_ids))
        log_likelihood = self.crf(emissions, labels, mask=mask.bool(), reduction='mean')
        return -log_likelihood

lstm_model = LSTMTagger(
    len(vocab), len(char_vocab), len(label_names), CUSTOM_HYPERPARAMS, pretrained_embedding_matrix
).to(DEVICE)
bilstm_model = BiLSTMTagger(
    len(vocab), len(char_vocab), len(label_names), CUSTOM_HYPERPARAMS, pretrained_embedding_matrix
).to(DEVICE)
bilstm_crf_model = BiLSTMCRFTagger(
    len(vocab), len(char_vocab), len(label_names), CUSTOM_HYPERPARAMS, pretrained_embedding_matrix
).to(DEVICE)

lstm_optimizer = torch.optim.Adam(
    lstm_model.parameters(),
    lr=CUSTOM_HYPERPARAMS['learning_rate'],
    weight_decay=CUSTOM_HYPERPARAMS['weight_decay'],
)
bilstm_optimizer = torch.optim.Adam(
    bilstm_model.parameters(),
    lr=CUSTOM_HYPERPARAMS['learning_rate'],
    weight_decay=CUSTOM_HYPERPARAMS['weight_decay'],
)
bilstm_crf_optimizer = torch.optim.Adam(
    bilstm_crf_model.parameters(),
    lr=CUSTOM_HYPERPARAMS['learning_rate'],
    weight_decay=CUSTOM_HYPERPARAMS['weight_decay'],
)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

lstm_model, bilstm_model, bilstm_crf_model

(LSTMTagger(
   (encoder): LSTMEncoder(
     (word_embedding): Embedding(21011, 300, padding_idx=0)
     (char_encoder): CharBiLSTMEncoder(
       (char_embedding): Embedding(86, 32, padding_idx=0)
       (char_lstm): LSTM(32, 64, batch_first=True, bidirectional=True)
     )
     (dropout): Dropout(p=0.3, inplace=False)
     (lstm): LSTM(428, 512, num_layers=2, batch_first=True, dropout=0.3)
   )
   (classifier): Linear(in_features=512, out_features=9, bias=True)
 ),
 BiLSTMTagger(
   (encoder): LSTMEncoder(
     (word_embedding): Embedding(21011, 300, padding_idx=0)
     (char_encoder): CharBiLSTMEncoder(
       (char_embedding): Embedding(86, 32, padding_idx=0)
       (char_lstm): LSTM(32, 64, batch_first=True, bidirectional=True)
     )
     (dropout): Dropout(p=0.3, inplace=False)
     (lstm): LSTM(428, 512, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
   )
   (classifier): Linear(in_features=1024, out_features=9, bias=True)
 ),
 BiLSTMCRFTagger(
   (encoder): L

## 9. Checkpoint Saving

In [11]:
def print_epoch_summary(model_name, epoch, total_epochs, train_loss, valid_loss, valid_f1=None):
    message = (
        f'{model_name} | Epoch {epoch}/{total_epochs} | '
        f'train_loss: {train_loss:.4f} | valid_loss: {valid_loss:.4f}'
    )
    if valid_f1 is not None:
        message += f' | valid_f1: {valid_f1:.4f}'
    print(message)

def save_custom_checkpoint(
    model_name,
    model,
    optimizer,
    epoch,
    history,
    best_valid_loss,
    patience_counter,
    is_best=False,
):
    checkpoint = {
        'model_name': model_name,
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'history': history,
        'best_valid_loss': best_valid_loss,
        'early_stopping_patience_counter': patience_counter,
        'vocab': vocab,
        'char_vocab': char_vocab,
        'label2id': label2id,
        'id2label': id2label,
        'hyperparameters': CUSTOM_HYPERPARAMS,
    }
    suffix = 'best' if is_best else 'last'
    checkpoint_path = CHECKPOINT_DIR / f'{model_name}_{suffix}.pt'
    torch.save(checkpoint, checkpoint_path)
    return checkpoint_path

def save_transformer_checkpoint(
    model,
    optimizer,
    epoch,
    history,
    best_valid_loss,
    patience_counter,
    scheduler=None,
    scaler=None,
    is_best=False,
):
    checkpoint = {
        'model_name': 'BERT',
        'base_model_name': TRANSFORMER_MODEL_NAME,
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'amp_scaler_state_dict': scaler.state_dict() if scaler is not None else None,
        'history': history,
        'best_valid_loss': best_valid_loss,
        'early_stopping_patience_counter': patience_counter,
        'label2id': label2id,
        'id2label': id2label,
        'hyperparameters': TRANSFORMER_HYPERPARAMS,
    }
    suffix = 'best' if is_best else 'last'
    checkpoint_path = CHECKPOINT_DIR / f'BERT_{suffix}.pt'
    torch.save(checkpoint, checkpoint_path)
    tokenizer.save_pretrained(CHECKPOINT_DIR / 'BERT_tokenizer')
    tokenizer.save_pretrained(CHECKPOINT_DIR / 'transformer_ner_tokenizer')
    return checkpoint_path

def save_full_comparison_checkpoint(model_results, histories=None):
    checkpoint = {
        'project': 'Named Entity Recognition System',
        'models': {
            'LSTM': {
                'model_state_dict': lstm_model.state_dict(),
                'optimizer_state_dict': lstm_optimizer.state_dict(),
            },
            'BiLSTM': {
                'model_state_dict': bilstm_model.state_dict(),
                'optimizer_state_dict': bilstm_optimizer.state_dict(),
            },
            'BiLSTM_CRF': {
                'model_state_dict': bilstm_crf_model.state_dict(),
                'optimizer_state_dict': bilstm_crf_optimizer.state_dict(),
            },
            'BERT': {
                'base_model_name': TRANSFORMER_MODEL_NAME,
                'model_state_dict': transformer_model.state_dict(),
                'optimizer_state_dict': transformer_optimizer.state_dict(),
                'scheduler_state_dict': transformer_scheduler.state_dict(),
                'amp_scaler_state_dict': amp_scaler.state_dict(),
            },
        },
        'histories': histories or {},
        'model_results': model_results,
        'vocab': vocab,
        'char_vocab': char_vocab,
        'label2id': label2id,
        'id2label': id2label,
        'custom_hyperparameters': CUSTOM_HYPERPARAMS,
        'transformer_hyperparameters': TRANSFORMER_HYPERPARAMS,
    }
    checkpoint_path = CHECKPOINT_DIR / 'lstm_bilstm_crf_bert_ner_comparison_checkpoint.pt'
    torch.save(checkpoint, checkpoint_path)
    return checkpoint_path

# Example calls inside each training loop:
# print_epoch_summary('LSTM', epoch, CUSTOM_HYPERPARAMS['max_epochs'], train_loss, valid_loss, valid_f1)
# save_custom_checkpoint('LSTM', lstm_model, lstm_optimizer, epoch, lstm_history, best_valid_loss, patience_counter, is_best=True)
# save_custom_checkpoint('BiLSTM', bilstm_model, bilstm_optimizer, epoch, bilstm_history, best_valid_loss, patience_counter, is_best=True)
# save_custom_checkpoint('BiLSTM_CRF', bilstm_crf_model, bilstm_crf_optimizer, epoch, crf_history, best_valid_loss, patience_counter, is_best=True)
# save_transformer_checkpoint(transformer_model, transformer_optimizer, epoch, bert_history, best_valid_loss, patience_counter, transformer_scheduler, amp_scaler, is_best=True)
# save_full_comparison_checkpoint(model_results, histories={'LSTM': lstm_history, 'BiLSTM': bilstm_history, 'BiLSTM_CRF': crf_history, 'BERT': bert_history})


## 10. Prepare Custom Model DataLoaders


In [12]:
class CustomNERDataset(Dataset):
    def __init__(self, split_dataset, vocab, char_vocab, max_sequence_length, max_char_length):
        self.split_dataset = split_dataset
        self.vocab = vocab
        self.char_vocab = char_vocab
        self.max_sequence_length = max_sequence_length
        self.max_char_length = max_char_length

    def __len__(self):
        return len(self.split_dataset)

    def encode_token_chars(self, token):
        char_ids = [self.char_vocab.get(char, self.char_vocab[UNK_TOKEN]) for char in token[:self.max_char_length]]
        char_ids += [self.char_vocab[PAD_TOKEN]] * (self.max_char_length - len(char_ids))
        return char_ids

    def __getitem__(self, index):
        example = self.split_dataset[index]
        tokens = example['tokens'][:self.max_sequence_length]
        labels = example['ner_tags'][:self.max_sequence_length]

        word_ids = [self.vocab.get(token.lower(), self.vocab[UNK_TOKEN]) for token in tokens]
        char_ids = [self.encode_token_chars(token) for token in tokens]
        mask = [1] * len(tokens)

        pad_length = self.max_sequence_length - len(tokens)
        word_ids += [self.vocab[PAD_TOKEN]] * pad_length
        char_ids += [[self.char_vocab[PAD_TOKEN]] * self.max_char_length for _ in range(pad_length)]
        labels += [-100] * pad_length
        mask += [0] * pad_length

        return {
            'word_ids': torch.tensor(word_ids, dtype=torch.long),
            'char_ids': torch.tensor(char_ids, dtype=torch.long),
            'labels': torch.tensor(labels, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.bool),
        }

custom_train_dataset = CustomNERDataset(
    dataset['train'], vocab, char_vocab,
    CUSTOM_HYPERPARAMS['max_sequence_length'],
    CUSTOM_HYPERPARAMS['max_char_length'],
)
custom_valid_dataset = CustomNERDataset(
    dataset['validation'], vocab, char_vocab,
    CUSTOM_HYPERPARAMS['max_sequence_length'],
    CUSTOM_HYPERPARAMS['max_char_length'],
)
custom_test_dataset = CustomNERDataset(
    dataset['test'], vocab, char_vocab,
    CUSTOM_HYPERPARAMS['max_sequence_length'],
    CUSTOM_HYPERPARAMS['max_char_length'],
)

custom_train_loader = DataLoader(custom_train_dataset, batch_size=CUSTOM_HYPERPARAMS['batch_size'], shuffle=True)
custom_valid_loader = DataLoader(custom_valid_dataset, batch_size=CUSTOM_HYPERPARAMS['batch_size'], shuffle=False)
custom_test_loader = DataLoader(custom_test_dataset, batch_size=CUSTOM_HYPERPARAMS['batch_size'], shuffle=False)

len(custom_train_dataset), len(custom_valid_dataset), len(custom_test_dataset)


(14041, 3250, 3453)

## 11. Train and Print Custom Model Outputs


In [13]:
def move_batch_to_device(batch):
    return {key: value.to(DEVICE) for key, value in batch.items()}

def train_one_custom_epoch(model, data_loader, optimizer, is_crf=False):
    model.train()
    total_loss = 0.0

    for batch in data_loader:
        batch = move_batch_to_device(batch)
        optimizer.zero_grad()

        if is_crf:
            crf_labels = batch['labels'].masked_fill(batch['labels'].eq(-100), 0)
            loss = model.loss(batch['word_ids'], batch['char_ids'], crf_labels, batch['mask'])
        else:
            logits = model(batch['word_ids'], batch['char_ids'])
            loss = criterion(logits.view(-1, len(label_names)), batch['labels'].view(-1))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CUSTOM_HYPERPARAMS['gradient_clip'])
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(data_loader)

def evaluate_custom_model(model, data_loader, is_crf=False):
    model.eval()
    total_loss = 0.0
    true_labels = []
    predicted_labels = []

    with torch.no_grad():
        for batch in data_loader:
            batch = move_batch_to_device(batch)

            if is_crf:
                crf_labels = batch['labels'].masked_fill(batch['labels'].eq(-100), 0)
                loss = model.loss(batch['word_ids'], batch['char_ids'], crf_labels, batch['mask'])
                decoded_predictions = model(batch['word_ids'], batch['char_ids'], batch['mask'])
            else:
                logits = model(batch['word_ids'], batch['char_ids'])
                loss = criterion(logits.view(-1, len(label_names)), batch['labels'].view(-1))
                decoded_predictions = torch.argmax(logits, dim=-1).cpu().tolist()

            total_loss += loss.item()
            labels = batch['labels'].cpu().tolist()
            masks = batch['mask'].cpu().tolist()

            for row_index, label_row in enumerate(labels):
                sentence_true = []
                sentence_pred = []
                token_count = sum(masks[row_index])
                prediction_row = decoded_predictions[row_index]

                for token_index in range(token_count):
                    label_id = label_row[token_index]
                    if label_id == -100:
                        continue
                    sentence_true.append(id2label[int(label_id)])
                    sentence_pred.append(id2label[int(prediction_row[token_index])])

                true_labels.append(sentence_true)
                predicted_labels.append(sentence_pred)

    return total_loss / len(data_loader), true_labels, predicted_labels

def show_classification_report(model_name, true_labels, predicted_labels):
    print(f'\n{model_name} classification report')
    print(classification_report(true_labels, predicted_labels, digits=4))
    return {
        'model': model_name,
        'precision': precision_score(true_labels, predicted_labels),
        'recall': recall_score(true_labels, predicted_labels),
        'f1': f1_score(true_labels, predicted_labels),
        'accuracy': accuracy_score(true_labels, predicted_labels),
    }

def train_custom_model(model_name, model, optimizer, is_crf=False):
    history = []
    best_valid_loss = float('inf')
    patience_counter = 0

    print(f'\nTraining {model_name}')
    for epoch in range(1, CUSTOM_HYPERPARAMS['max_epochs'] + 1):
        train_loss = train_one_custom_epoch(model, custom_train_loader, optimizer, is_crf=is_crf)
        valid_loss, valid_true, valid_pred = evaluate_custom_model(model, custom_valid_loader, is_crf=is_crf)
        valid_f1 = f1_score(valid_true, valid_pred)

        print_epoch_summary(
            model_name,
            epoch,
            CUSTOM_HYPERPARAMS['max_epochs'],
            train_loss,
            valid_loss,
            valid_f1,
        )

        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'valid_loss': valid_loss,
            'valid_f1': valid_f1,
        })

        improved = valid_loss < best_valid_loss - CUSTOM_HYPERPARAMS['min_improvement']
        if improved:
            best_valid_loss = valid_loss
            patience_counter = 0
            save_custom_checkpoint(model_name, model, optimizer, epoch, history, best_valid_loss, patience_counter, is_best=True)
        else:
            patience_counter += 1

        save_custom_checkpoint(model_name, model, optimizer, epoch, history, best_valid_loss, patience_counter, is_best=False)

        if patience_counter >= CUSTOM_HYPERPARAMS['early_stopping_patience']:
            print(f'{model_name} early stopping at epoch {epoch}')
            break

    test_loss, test_true, test_pred = evaluate_custom_model(model, custom_test_loader, is_crf=is_crf)
    print(f'{model_name} test_loss: {test_loss:.4f}')
    result = show_classification_report(model_name, test_true, test_pred)
    return history, result

model_results = []

lstm_history, lstm_result = train_custom_model('LSTM', lstm_model, lstm_optimizer, is_crf=False)
model_results.append(lstm_result)

bilstm_history, bilstm_result = train_custom_model('BiLSTM', bilstm_model, bilstm_optimizer, is_crf=False)
model_results.append(bilstm_result)

crf_history, crf_result = train_custom_model('BiLSTM_CRF', bilstm_crf_model, bilstm_crf_optimizer, is_crf=True)
model_results.append(crf_result)

pd.DataFrame(model_results).sort_values('f1', ascending=False)



Training LSTM
LSTM | Epoch 1/50 | train_loss: 0.6680 | valid_loss: 0.3373 | valid_f1: 0.4740
LSTM | Epoch 2/50 | train_loss: 0.2114 | valid_loss: 0.1535 | valid_f1: 0.7410
LSTM | Epoch 3/50 | train_loss: 0.1294 | valid_loss: 0.1363 | valid_f1: 0.7450
LSTM | Epoch 4/50 | train_loss: 0.1074 | valid_loss: 0.1071 | valid_f1: 0.7935
LSTM | Epoch 5/50 | train_loss: 0.0963 | valid_loss: 0.1012 | valid_f1: 0.7993
LSTM | Epoch 6/50 | train_loss: 0.0866 | valid_loss: 0.1011 | valid_f1: 0.8091
LSTM | Epoch 7/50 | train_loss: 0.0797 | valid_loss: 0.1038 | valid_f1: 0.8031
LSTM | Epoch 8/50 | train_loss: 0.0708 | valid_loss: 0.1000 | valid_f1: 0.8087
LSTM | Epoch 9/50 | train_loss: 0.0678 | valid_loss: 0.1006 | valid_f1: 0.7911
LSTM | Epoch 10/50 | train_loss: 0.0622 | valid_loss: 0.1051 | valid_f1: 0.7867
LSTM | Epoch 11/50 | train_loss: 0.0574 | valid_loss: 0.0887 | valid_f1: 0.8224
LSTM | Epoch 12/50 | train_loss: 0.0543 | valid_loss: 0.0908 | valid_f1: 0.8223
LSTM | Epoch 13/50 | train_loss: 0

,model,precision,recall,f1,accuracy
2,BiLSTM_CRF,0.848615,0.840652,0.844614,0.969290
1,BiLSTM,0.819678,0.820113,0.819896,0.967826
0,LSTM,0.734110,0.758676,0.746191,0.958092


## 12. Fine-Tune Transformer

In [14]:
def align_predictions(predictions, label_ids):
    pred_ids = np.argmax(predictions, axis=2)
    true_labels = []
    true_predictions = []

    for prediction_row, label_row in zip(pred_ids, label_ids):
        sentence_labels = []
        sentence_predictions = []
        for prediction_id, label_id in zip(prediction_row, label_row):
            if label_id == -100:
                continue
            sentence_labels.append(id2label[int(label_id)])
            sentence_predictions.append(id2label[int(prediction_id)])
        true_labels.append(sentence_labels)
        true_predictions.append(sentence_predictions)

    return true_labels, true_predictions

def compute_ner_metrics(eval_pred):
    logits, labels = eval_pred
    true_labels, true_predictions = align_predictions(logits, labels)
    return {
        'precision': precision_score(true_labels, true_predictions),
        'recall': recall_score(true_labels, true_predictions),
        'f1': f1_score(true_labels, true_predictions),
        'accuracy': accuracy_score(true_labels, true_predictions),
    }

class EpochPrinterCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, **kwargs):
        epoch = int(state.epoch or 0) + 1
        total_epochs = int(args.num_train_epochs)
        print(f'\nEpoch {epoch}/{total_epochs}')

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not metrics:
            return
        parts = []
        for name in ['eval_loss', 'eval_f1', 'eval_accuracy']:
            value = metrics.get(name)
            if value is not None:
                parts.append(f'{name}: {value:.4f}')
        if parts:
            print('Validation - ' + ', '.join(parts))

transformer_model = AutoModelForTokenClassification.from_pretrained(
    TRANSFORMER_MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)
transformer_model = transformer_model.to(DEVICE)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR / 'bert_trainer_outputs'),
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=TRANSFORMER_HYPERPARAMS['learning_rate'],
    per_device_train_batch_size=TRANSFORMER_HYPERPARAMS['batch_size'],
    per_device_eval_batch_size=TRANSFORMER_HYPERPARAMS['batch_size'],
    num_train_epochs=TRANSFORMER_HYPERPARAMS['max_epochs'],
    weight_decay=TRANSFORMER_HYPERPARAMS['weight_decay'],
    warmup_ratio=TRANSFORMER_HYPERPARAMS['warmup_ratio'],
    fp16=TRANSFORMER_HYPERPARAMS['mixed_precision_amp'] and torch.cuda.is_available(),
    logging_strategy='epoch',
    disable_tqdm=True,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    report_to='none',
)

transformer_optimizer = torch.optim.AdamW(
    transformer_model.parameters(),
    lr=TRANSFORMER_HYPERPARAMS['learning_rate'],
    weight_decay=TRANSFORMER_HYPERPARAMS['weight_decay'],
)
steps_per_epoch = int(np.ceil(len(tokenized_dataset['train']) / TRANSFORMER_HYPERPARAMS['batch_size']))
total_training_steps = steps_per_epoch * TRANSFORMER_HYPERPARAMS['max_epochs']
warmup_steps = int(total_training_steps * TRANSFORMER_HYPERPARAMS['warmup_ratio'])
transformer_scheduler = get_linear_schedule_with_warmup(
    transformer_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)
amp_scaler = torch.cuda.amp.GradScaler(enabled=training_args.fp16)

trainer = Trainer(
    model=transformer_model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_ner_metrics,
    callbacks=[EpochPrinterCallback()],
    optimizers=(transformer_optimizer, transformer_scheduler),
)

transformer_model

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 5955.24it/s]
BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expec

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

## 13. Train and Print Transformer Output


In [15]:
if 'model_results' not in globals():
    model_results = []

print('\nTraining BERT')
bert_train_output = trainer.train()

bert_valid_predictions = trainer.predict(tokenized_dataset['validation'])
y_true_bert_valid, y_pred_bert_valid = align_predictions(
    bert_valid_predictions.predictions,
    bert_valid_predictions.label_ids,
)
bert_eval_metrics = {
    'eval_precision': precision_score(y_true_bert_valid, y_pred_bert_valid),
    'eval_recall': recall_score(y_true_bert_valid, y_pred_bert_valid),
    'eval_f1': f1_score(y_true_bert_valid, y_pred_bert_valid),
    'eval_accuracy': accuracy_score(y_true_bert_valid, y_pred_bert_valid),
}

print('\nBERT validation metrics')
for metric_name, metric_value in bert_eval_metrics.items():
    print(f'{metric_name}: {metric_value:.4f}')

bert_predictions = trainer.predict(tokenized_dataset['test'])
y_true_bert, y_pred_bert = align_predictions(bert_predictions.predictions, bert_predictions.label_ids)
bert_result = show_classification_report('BERT', y_true_bert, y_pred_bert)
model_results.append(bert_result)

bert_history = trainer.state.log_history
bert_valid_losses = [row['eval_loss'] for row in bert_history if 'eval_loss' in row]
bert_best_valid_loss = min(bert_valid_losses) if bert_valid_losses else None
save_transformer_checkpoint(
    transformer_model,
    transformer_optimizer,
    int(TRANSFORMER_HYPERPARAMS['max_epochs']),
    bert_history,
    bert_best_valid_loss,
    patience_counter=0,
    scheduler=transformer_scheduler,
    scaler=amp_scaler,
    is_best=True,
)
transformer_model.save_pretrained(CHECKPOINT_DIR / 'transformer_ner_model')
tokenizer.save_pretrained(CHECKPOINT_DIR / 'transformer_ner_tokenizer')

bert_result



Training BERT

Epoch 1/5
{'loss': '0.4574', 'grad_norm': '2.181', 'learning_rate': '1.781e-05', 'epoch': '1'}
{'eval_loss': '0.07884', 'eval_precision': '0.8961', 'eval_recall': '0.912', 'eval_f1': '0.904', 'eval_accuracy': '0.9771', 'eval_runtime': '1.954', 'eval_samples_per_second': '1663', 'eval_steps_per_second': '52.19', 'epoch': '1'}
Validation - eval_loss: 0.0788, eval_f1: 0.9040, eval_accuracy: 0.9771


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.11it/s]



Epoch 2/5
{'loss': '0.06266', 'grad_norm': '1.34', 'learning_rate': '1.337e-05', 'epoch': '2'}
{'eval_loss': '0.06667', 'eval_precision': '0.9226', 'eval_recall': '0.9271', 'eval_f1': '0.9249', 'eval_accuracy': '0.982', 'eval_runtime': '1.87', 'eval_samples_per_second': '1738', 'eval_steps_per_second': '54.54', 'epoch': '2'}
Validation - eval_loss: 0.0667, eval_f1: 0.9249, eval_accuracy: 0.9820


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.95it/s]



Epoch 3/5
{'loss': '0.03296', 'grad_norm': '1.607', 'learning_rate': '8.927e-06', 'epoch': '3'}
{'eval_loss': '0.05987', 'eval_precision': '0.9355', 'eval_recall': '0.9406', 'eval_f1': '0.938', 'eval_accuracy': '0.9846', 'eval_runtime': '1.85', 'eval_samples_per_second': '1757', 'eval_steps_per_second': '55.14', 'epoch': '3'}
Validation - eval_loss: 0.0599, eval_f1: 0.9380, eval_accuracy: 0.9846


...

{'train_runtime': '150.8', 'train_samples_per_second': '465.4', 'train_steps_per_second': '14.55', 'train_loss': '0.1177', 'epoch': '5'}


There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


BERT validation metrics
eval_precision: 0.9393
eval_recall: 0.9454
eval_f1: 0.9423
eval_accuracy: 0.9856

BERT classification report
              precision    recall  f1-score   support

         LOC     0.9206    0.9193    0.9199      3000
        MISC     0.7078    0.7385    0.7228      1266
         ORG     0.8855    0.9279    0.9062      3524
         PER     0.9503    0.9411    0.9457      2989

   micro avg     0.8912    0.9069    0.8990     10779
   macro avg     0.8660    0.8817    0.8737     10779
weighted avg     0.8924    0.9069    0.8994     10779



Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.90it/s]


{'model': 'BERT',
 'precision': 0.8911577028258888,
 'recall': 0.9069486965395677,
 'f1': 0.8989838613269576,
 'accuracy': 0.9713209687839776}

The transformer used in this project is `bert-base-cased`. It is better than an uncased BERT baseline for NER because capitalization is an important signal for names, organizations, and places.

Use `TrainingArguments`, `Trainer`, and the `seqeval` metric functions below. Save the final model with `save_pretrained('checkpoints/transformer_ner_model')` and save the tokenizer to `checkpoints/transformer_ner_tokenizer`.

## 14. Evaluation and Model Comparison

In [16]:
comparison_df = pd.DataFrame(model_results).sort_values('f1', ascending=False) if model_results else pd.DataFrame(
    columns=['model', 'precision', 'recall', 'f1', 'accuracy']
)
comparison_df


,model,precision,recall,f1,accuracy
3,BERT,0.891158,0.906949,0.898984,0.971321
2,BiLSTM_CRF,0.848615,0.840652,0.844614,0.969290
1,BiLSTM,0.819678,0.820113,0.819896,0.967826
0,LSTM,0.734110,0.758676,0.746191,0.958092


### Why the CRF improves boundary detection

The BiLSTM + CRF achieved a higher test F1 score (0.8446) than the standard BiLSTM (0.8199), an improvement of about 2.47 percentage points. A standard BiLSTM predicts each token label independently, while the CRF learns transition scores between consecutive IOB labels and decodes the best label sequence for the complete sentence. This can discourage inconsistent transitions, such as `I-ORG` appearing without a suitable `B-ORG`, and helps produce more coherent entity boundaries.

## 15. Test on New Text

In [17]:
sample_tokens = ['Peter', 'Blackburn', 'works', 'for', 'the', 'European', 'Commission', 'in', 'Brussels', '.']
sample_tokens

['Peter',
 'Blackburn',
 'works',
 'for',
 'the',
 'European',
 'Commission',
 'in',
 'Brussels',
 '.']